In [1]:
from pyspark.sql.functions import *

StatementMeta(, ff139321-69c7-4da2-8bcf-f98d30dc4f88, 3, Finished, Available, Finished, False)

In [2]:
def read_table(table_name):
    return spark.read.table(f"Bronze_lakehouse.dbo.{table_name}")

StatementMeta(, ff139321-69c7-4da2-8bcf-f98d30dc4f88, 4, Finished, Available, Finished, False)

In [5]:
def save_table(df, table_name):
    df.write.format("delta").mode("overwrite").saveAsTable(f"Silver_lakehouse.dbo.{table_name}")

StatementMeta(, ff139321-69c7-4da2-8bcf-f98d30dc4f88, 7, Finished, Available, Finished, False)

In [3]:
def transform_taxi_zones(df):
    df = df.replace("N/A", None)
    df = df.withColumn("Borough", when(col("Zone").isin("Unknown", "Outside of NYC"), col("Zone")).otherwise(col("Borough")))
    df = df.withColumn("Zone", when(col("Zone").isin("Unknown", "Outside of NYC"), None).otherwise(col("Zone")))
    return df

StatementMeta(, ff139321-69c7-4da2-8bcf-f98d30dc4f88, 5, Finished, Available, Finished, False)

In [10]:
def validate_taxi_zones(df):
    assert df.count() > 0, "Table is empty"
    assert df.filter(col("LocationID").isNull()).count() == 0, "LocationID has null values"
    assert df.groupBy("LocationID").count().filter(col("count") > 1).count() == 0, "Duplicate LocationID in table"
    for column in ["Borough", "Zone", "service_zone"]:
        assert df.filter(trim(col(column)) == "").count() == 0, f"{column} contains empty strings"
    return True

StatementMeta(, ff139321-69c7-4da2-8bcf-f98d30dc4f88, 12, Finished, Available, Finished, False)

In [11]:
def process_taxi_zones():
    df = read_table("NYC_Taxi_zones")
    df = transform_taxi_zones(df)
    if validate_taxi_zones(df):
        save_table(df, "NYC_Taxi_zones")

StatementMeta(, ff139321-69c7-4da2-8bcf-f98d30dc4f88, 13, Finished, Available, Finished, False)

In [12]:
process_taxi_zones()

StatementMeta(, ff139321-69c7-4da2-8bcf-f98d30dc4f88, 14, Finished, Available, Finished, True)